In [1]:
import os
import requests
import json
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("SEOUL_API_KEY")

# URL을 코드에서 조립 (sample 자리에 API_KEY, xml → json)
#url = f"http://openapi.seoul.go.kr:8088/{API_KEY}/json/CardSubwayStatsNew/1/5/20220301"
url = f"http://openapi.seoul.go.kr:8088/{API_KEY}/json/culturalEventInfo/1/100"
print("요청 URL:", url)  # 디버깅용

response = requests.get(url)
print("상태코드:", response.status_code)

data = response.json()
data

요청 URL: http://openapi.seoul.go.kr:8088/4e4666566e73656c38314a4e4d6345/json/culturalEventInfo/1/100
상태코드: 200


{'culturalEventInfo': {'list_total_count': 3928,
  'RESULT': {'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다'},
  'row': [{'CODENAME': '전시/미술',
    'GUNAME': '강남구',
    'TITLE': '해외바이어가 직접 찾는 글로벌 전시회 [2026 인터참코리아] InterCHARM Korea',
    'DATE': '2026-07-01~2026-07-03',
    'PLACE': '코엑스 A홀, C홀 ',
    'ORG_NAME': '기타',
    'USE_TRGT': '뷰티산업 종사자',
    'USE_FEE': '현장등록: 20,000원 (사전 등록 시, 무료 입장 가능)',
    'INQUIRY': '070-5095-9904 / 9903',
    'PLAYER': '',
    'PROGRAM': '',
    'ETC_DESC': '',
    'ORG_LINK': 'https://www.intercharmkorea.com/ko-kr.html',
    'MAIN_IMG': 'https://culture.seoul.go.kr/cmmn/file/getImage.do?atchFileId=5f2a4e2e793e49b987e5902a6e58cdf7&thumb=Y',
    'RGSTDATE': '2026-01-20',
    'TICKET': '시민',
    'STRTDATE': '2026-07-01 00:00:00.0',
    'END_DATE': '2026-07-03 00:00:00.0',
    'THEMECODE': '기타',
    'LOT': '127.059159043842',
    'LAT': '37.5118239121138',
    'IS_FREE': '유료',
    'HMPG_ADDR': 'https://culture.seoul.go.kr/culture/culture/cultureEvent/view.do?cult

In [2]:
BASE_URL = f"http://openapi.seoul.go.kr:8088/{API_KEY}/json/culturalEventInfo"
response = requests.get(f"{BASE_URL}/1/1")
total = response.json()["culturalEventInfo"]["list_total_count"]
print(f"전체: {total}건")

전체: 3928건


In [3]:
# 1000건씩 반복 호출
all_rows = []
for start in range(1, total + 1, 1000):
    end = min(start + 999, total)
    url = f"{BASE_URL}/{start}/{end}"
    res = requests.get(url)
    rows = res.json()["culturalEventInfo"]["row"]
    all_rows.extend(rows)
    print(f"{start}~{end} 수신 ({len(rows)}건)")
    
# 파일 저장
with open("cultural_events.json", "w", encoding="utf-8") as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)

print("저장 완료: cultural_events.json")

1~1000 수신 (1000건)
1001~2000 수신 (1000건)
2001~3000 수신 (1000건)
3001~3928 수신 (928건)
저장 완료: cultural_events.json


In [4]:
with open("cultural_events.json", "r", encoding="utf-8") as f:
    rows = json.load(f)

rows

[{'CODENAME': '전시/미술',
  'GUNAME': '강남구',
  'TITLE': '해외바이어가 직접 찾는 글로벌 전시회 [2026 인터참코리아] InterCHARM Korea',
  'DATE': '2026-07-01~2026-07-03',
  'PLACE': '코엑스 A홀, C홀 ',
  'ORG_NAME': '기타',
  'USE_TRGT': '뷰티산업 종사자',
  'USE_FEE': '현장등록: 20,000원 (사전 등록 시, 무료 입장 가능)',
  'INQUIRY': '070-5095-9904 / 9903',
  'PLAYER': '',
  'PROGRAM': '',
  'ETC_DESC': '',
  'ORG_LINK': 'https://www.intercharmkorea.com/ko-kr.html',
  'MAIN_IMG': 'https://culture.seoul.go.kr/cmmn/file/getImage.do?atchFileId=5f2a4e2e793e49b987e5902a6e58cdf7&thumb=Y',
  'RGSTDATE': '2026-01-20',
  'TICKET': '시민',
  'STRTDATE': '2026-07-01 00:00:00.0',
  'END_DATE': '2026-07-03 00:00:00.0',
  'THEMECODE': '기타',
  'LOT': '127.059159043842',
  'LAT': '37.5118239121138',
  'IS_FREE': '유료',
  'HMPG_ADDR': 'https://culture.seoul.go.kr/culture/culture/cultureEvent/view.do?cultcode=156571&menuNo=200009',
  'PRO_TIME': '10:00~17:00'},
 {'CODENAME': '클래식',
  'GUNAME': '강남구',
  'TITLE': 'GS아트센터 x 국립심포니오케스트라 라이브 애니메이션 시네스테틱스 [피터와 늑대 & 어미 거

In [5]:
!pip uninstall psycopg2 -y
!pip install psycopg2-binary


In [1]:
import os
import json
import psycopg2
from dotenv import load_dotenv

load_dotenv()

# JSON 파일 불러오기
with open("cultural_events.json", "r", encoding="utf-8") as f:
    rows = json.load(f)
DB_PASSWORD = os.getenv("DB_PASSWORD")
# DB 연결
conn = psycopg2.connect(
    host="localhost",
    dbname="postgres",
    user="postgres",
    password=DB_PASSWORD,
    port=5432,
    # options="-c client_encoding=UTF8"
)
cur = conn.cursor()

# 테이블 생성
cur.execute("""
    CREATE TABLE IF NOT EXISTS cultural_events (
        id SERIAL PRIMARY KEY,
        codename TEXT,
        guname TEXT,
        title TEXT,
        date TEXT,
        place TEXT,
        org_name TEXT,
        use_trgt TEXT,
        use_fee TEXT,
        inquiry TEXT,
        player TEXT,
        program TEXT,
        etc_desc TEXT,
        org_link TEXT,
        main_img TEXT,
        rgstdate TEXT,
        ticket TEXT,
        strtdate TEXT,
        end_date TEXT,
        themecode TEXT,
        lot TEXT,
        lat TEXT,
        is_free TEXT,
        hmpg_addr TEXT,
        pro_time TEXT
    )
""")

# 저장
for row in rows:
    cur.execute("""
        INSERT INTO cultural_events (
            codename, guname, title, date, place, org_name, use_trgt, use_fee,
            inquiry, player, program, etc_desc, org_link, main_img, rgstdate,
            ticket, strtdate, end_date, themecode, lot, lat, is_free, hmpg_addr, pro_time
        ) VALUES (
            %(CODENAME)s, %(GUNAME)s, %(TITLE)s, %(DATE)s, %(PLACE)s, %(ORG_NAME)s,
            %(USE_TRGT)s, %(USE_FEE)s, %(INQUIRY)s, %(PLAYER)s, %(PROGRAM)s,
            %(ETC_DESC)s, %(ORG_LINK)s, %(MAIN_IMG)s, %(RGSTDATE)s, %(TICKET)s,
            %(STRTDATE)s, %(END_DATE)s, %(THEMECODE)s, %(LOT)s, %(LAT)s,
            %(IS_FREE)s, %(HMPG_ADDR)s, %(PRO_TIME)s
        )
    """, row)

conn.commit()
print(f"DB 저장 완료: {len(rows)}건")

# 확인
cur.execute("SELECT title, guname, is_free FROM cultural_events LIMIT 3")
for r in cur.fetchall():
    print(r)

cur.close()
conn.close()


DB 저장 완료: 3928건
('해외바이어가 직접 찾는 글로벌 전시회 [2026 인터참코리아] InterCHARM Korea', '강남구', '유료')
('GS아트센터 x 국립심포니오케스트라 라이브 애니메이션 시네스테틱스 [피터와 늑대 & 어미 거위]', '강남구', '유료')
('[마포문화재단] M 마티네 [2026 MAC 모닝 콘서트] ＃3', '마포구', '유료')
